# Phân tích cách chấm lại logits (cho bài báo, chỉ cần CPU)

Notebook trả lời 3 câu hỏi về cách chỉnh logits (logit adjustment + ghép verb–noun), trên ViT-L và ViT-g:
1. **Mức tăng đến từ đâu:** recall theo nhóm lớp (số lần xuất hiện trong train: many > 100, medium 20–100, few < 20), với 4 cách:
   probe Meta, chỉ logit adjustment, chỉ ghép verb–noun, cả hai. Kèm tỉ lệ ô top-5 mà mỗi nhóm chiếm, so với tỉ lệ nhãn thật.
2. **Vì sao τ=1 thất bại:** probe của Meta được train bằng **sigmoid focal loss** (α=0.25, γ=2), không phải softmax cross-entropy mà lý
   thuyết logit adjustment giả định. Notebook đo prior của tập train đã nằm sẵn trong logits bao nhiêu (độ dốc của logit trung bình mỗi lớp
   theo log prior) và độ hiệu chỉnh (ECE).
3. **Đường recall theo τ**, có và không có ghép verb–noun.

Tham số τ, β đều được chọn bằng cross-fitting theo người tham gia (2 fold), giống notebook `kaggle_posthoc_ek100`.

**Cài đặt:** Accelerator **None** (CPU), Internet On. *Add Input*:
- output của `kaggle-eval-ek100-original` (thư mục `logits_orig`, ViT-L);
- output **mới nhất** của notebook ViT-g phần 0 (`eval-vitg384-0`, có thư mục `merged_vitg`). Nếu không có `merged_vitg`, *Add Input*
  output của cả 2 phần (`logits_vitg_part0`, `logits_vitg_part1`), notebook sẽ tự ghép.

Mỗi thư mục logits mất khoảng 3–5 phút. Kết quả gồm `analysis.md` (bảng), `analysis.json`, `fig_tau_curve.png`, `fig_groups.png`.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!pip install -q decord matplotlib
R = "/kaggle/working/repo"
ANN = "/kaggle/working/ek100-ann"
# Clip / class index (same as in the eval runs; no videos needed)
!python $R/scripts/prepare_ek100.py --download_layout --video_root /kaggle/tmp/ek100 \
    --train_csv $ANN/EPIC_100_train.csv --val_csv $ANN/EPIC_100_validation.csv \
    --video_info $ANN/EPIC_100_video_info.csv --out_dir /kaggle/working/work | grep '"val_clips"'


In [ ]:
import glob, os, shutil
def first(pattern):
    found = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return found[0] if found else None

logits = {}
vitl = first("logits_orig")
if vitl:
    if os.path.isdir(f"{vitl}/official"): logits["ViT-L (giao thức gốc)"] = f"{vitl}/official"
    if os.path.isdir(f"{vitl}/action_start"): logits["ViT-L (action_start)"] = f"{vitl}/action_start"
vitg = first("merged_vitg")
if vitg is None:  # merge the two ViT-g parts here
    parts = {k: first(f"logits_vitg_part{k}") for k in (0, 1)}
    if all(parts.values()):
        vitg = "/kaggle/working/merged_vitg"
        for k, d in parts.items():
            for sd in glob.glob(f"{d}/*/shard*"):
                anchor, shard = sd.split("/")[-2], int(sd.split("shard")[-1])
                shutil.copytree(sd, f"{vitg}/{anchor}/shard{10 * k + shard}", dirs_exist_ok=True)
if vitg and os.path.isdir(f"{vitg}/official"):
    logits["ViT-g (giao thức gốc)"] = f"{vitg}/official"
assert logits, "no logits found: Add Input the outputs listed above"
for k, v in logits.items():
    print(f"{k:24s} {v}")


In [ ]:
ARGS = " ".join(f'"{k}={v}"' for k, v in logits.items())
!python $R/scripts/analyze_posthoc.py --work_dir /kaggle/working/work --train_csv $ANN/EPIC_100_train.csv \
    --logits $ARGS --out_dir /kaggle/working/analysis
from IPython.display import Image, display
display(Image("/kaggle/working/analysis/fig_groups.png"))
display(Image("/kaggle/working/analysis/fig_tau_curve.png"))


Gửi lại output của cell cuối (các bảng và 2 hình), hoặc tải thư mục `analysis/` trong tab Output.